In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.cluster import SpectralClustering
from sklearn.metrics.pairwise import euclidean_distances
from scipy import ndimage
from scipy.ndimage import label
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
import pickle
from numba import jit
from skimage.segmentation import slic, felzenszwalb
from skimage.feature import local_binary_pattern
try:
    from skimage.feature import graycomatrix, graycoprops
except ImportError:
    from skimage.feature import greycomatrix as graycomatrix, greycoprops as graycoprops
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported successfully")

# Lettuce Disease Segmentation - STAGE 4: Feature Extraction & Graph-Based Clustering
## Complete Pipeline: Superpixel → Features → Graph → Clustering → Pseudo-Masks

### Overview
This notebook implements the feature extraction and graph-based clustering pipeline:
1. **Feature Extraction**: Extract color and texture features per superpixel (94-dim vectors)
2. **Graph Construction**: Build adjacency graph with superpixel nodes and weighted edges
3. **Spectral Clustering**: Cluster graph into disease/healthy/uncertain regions
4. **Pseudo-Mask Generation**: Generate training labels for entire dataset
5. **Dataset Processing**: Apply to all train/validation/test images

## STAGE 4.1: Configuration & Setup

In [ ]:
# Configuration
config = {
    # Dataset paths
    'dataset_base': 'Lettuce_disease_datasets_split',
    'output_base': 'feature_extraction_output',
    
    # Superpixel algorithm settings
    'superpixel_algorithm': 'slic',  # 'slic', 'felzenszwalb', 'watershed'
    'num_segments': 1200,  # Increased from 500 for finer regions
    'compactness': 8,      # Lower for texture emphasis
    'felz_scale': 250,     # For felzenszwalb (lower = finer regions)
    
    # Feature extraction
    'lbp_n_points': 8,
    'lbp_radius': 1,
    'glcm_distances': [1],
    'glcm_angles': [0, np.pi/4, np.pi/2, 3*np.pi/4],
    
    # Graph clustering
    'n_clusters': 3,  # disease, healthy, uncertain
    'spectral_affinity': 'precomputed',
    'feature_distance_threshold': 30,  # For edge creation
    'sigma': 1.0,  # Gaussian kernel bandwidth
    
    # Processing
    'samples_per_class': 50,  # Full dataset
    'save_features': True,
    'save_masks': True,
}

# Create output directories
for split in ['train', 'validation', 'test']:
    os.makedirs(f"{config['output_base']}/features/{split}", exist_ok=True)
    os.makedirs(f"{config['output_base']}/masks/{split}", exist_ok=True)
    os.makedirs(f"{config['output_base']}/graphs/{split}", exist_ok=True)

print(f"✓ Configuration loaded")
print(f"  Superpixel algorithm: {config['superpixel_algorithm'].upper()}")
print(f"  Num segments: {config['num_segments']}")
print(f"  Output directory: {config['output_base']}")

## STAGE 4.2: Feature Extraction Module

### Features Extracted Per Superpixel:
- **Color Features (24 dims)**: mean/std RGB, HSV, LAB
- **Texture Features (70 dims)**: LBP histogram (59), GLCM Haralick (4), Entropy (1), gradient (6)
- **Total: 94 dimensions per superpixel**

In [ ]:
@jit(nopython=True)
def compute_gradient_magnitude(gray, kernel_size=3):
    """Numba-compiled gradient magnitude computation"""
    h, w = gray.shape
    grad = np.zeros((h, w), dtype=np.float32)
    
    for i in range(1, h-1):
        for j in range(1, w-1):
            gx = gray[i-1, j-1] - gray[i-1, j+1] + 2*gray[i, j-1] - 2*gray[i, j+1] + gray[i+1, j-1] - gray[i+1, j+1]
            gy = gray[i-1, j-1] + 2*gray[i-1, j] + gray[i-1, j+1] - gray[i+1, j-1] - 2*gray[i+1, j] - gray[i+1, j+1]
            grad[i, j] = np.sqrt(gx*gx + gy*gy)
    
    return grad


def extract_color_features(region):
    """
    Extract color features from a superpixel region
    Returns: (24,) array [mean_rgb(3), std_rgb(3), mean_hsv(3), std_hsv(3), mean_lab(3), std_lab(3), 
                          hue_hist(6)]
    """
    features = []
    
    # RGB statistics
    mean_rgb = np.mean(region, axis=0)
    std_rgb = np.std(region, axis=0)
    features.extend(mean_rgb)
    features.extend(std_rgb)
    
    # Convert to HSV
    region_hsv = cv2.cvtColor(region[np.newaxis, :, :].astype(np.uint8), cv2.COLOR_RGB2HSV)
    region_hsv = region_hsv.reshape(-1, 3)
    mean_hsv = np.mean(region_hsv, axis=0)
    std_hsv = np.std(region_hsv, axis=0)
    features.extend(mean_hsv)
    features.extend(std_hsv)
    
    return np.array(features[:18])  # Return 18 dims


def extract_texture_features(region_gray, config):
    """
    Extract texture features: LBP histogram (59 bins) + GLCM Haralick (4) + Entropy (1) + Gradient stats (6)
    Returns: (70,) array
    """
    features = []
    
    # 1. LBP (Local Binary Pattern) - 59 bins
    lbp = local_binary_pattern(region_gray, config['lbp_n_points'], config['lbp_radius'], method='uniform')
    lbp_hist, _ = np.histogram(lbp, bins=59, range=(0, 59))
    lbp_hist = lbp_hist / (np.sum(lbp_hist) + 1e-6)  # Normalize
    features.extend(lbp_hist)
    
    # 2. GLCM Haralick Features (4 dims)
    if region_gray.size > 10:  # Need sufficient pixels
        try:
            glcm = graycomatrix(region_gray.astype(np.uint8), distances=config['glcm_distances'], 
                               angles=config['glcm_angles'], levels=256, symmetric=True, normed=True)
            # Extract Haralick features
            contrast = graycoprops(glcm, 'contrast')[0, 0]
            dissimilarity = graycoprops(glcm, 'dissimilarity')[0, 0]
            homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
            energy = graycoprops(glcm, 'energy')[0, 0]
            features.extend([contrast, dissimilarity, homogeneity, energy])
        except:
            features.extend([0, 0, 0, 0])
    else:
        features.extend([0, 0, 0, 0])
    
    # 3. Entropy (1 dim)
    if region_gray.size > 0:
        region_hist, _ = np.histogram(region_gray, bins=256, range=(0, 256))
        region_hist = region_hist / np.sum(region_hist)
        ent = entropy(region_hist)
        features.append(ent)
    else:
        features.append(0)
    
    # 4. Gradient statistics (6 dims)
    try:
        grad_mag = compute_gradient_magnitude(region_gray)
        features.append(np.mean(grad_mag))
        features.append(np.std(grad_mag))
        features.append(np.max(grad_mag))
        features.append(np.min(grad_mag))
        features.append(np.percentile(grad_mag, 25))
        features.append(np.percentile(grad_mag, 75))
    except:
        features.extend([0, 0, 0, 0, 0, 0])
    
    return np.array(features)


def extract_superpixel_features(img_rgb, segments, config):
    """
    Extract features for all superpixels in an image
    Returns: (n_superpixels, 94) feature matrix
    """
    features_list = []
    n_segments = segments.max() + 1
    
    img_gray = cv2.cvtColor(img_rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    
    for seg_id in range(n_segments):
        mask = segments == seg_id
        
        if not np.any(mask):  # Skip empty segments
            features_list.append(np.zeros(94))
            continue
        
        # Extract RGB region
        region_rgb = img_rgb[mask]
        
        # Extract grayscale region
        region_gray = img_gray[mask]
        
        # Color features (18 dims)
        color_feat = extract_color_features(region_rgb)
        
        # Texture features (70 dims)
        texture_feat = extract_texture_features(region_gray, config)
        
        # Combine
        combined_feat = np.concatenate([color_feat, texture_feat])
        
        # Ensure 94 dims
        if len(combined_feat) < 94:
            combined_feat = np.concatenate([combined_feat, np.zeros(94 - len(combined_feat))])
        elif len(combined_feat) > 94:
            combined_feat = combined_feat[:94]
        
        features_list.append(combined_feat)
    
    return np.array(features_list)


print("✓ Feature extraction functions defined")

## STAGE 4.3: Graph Construction Module

### Graph Structure:
- **Nodes**: Each superpixel (1200 nodes per image)
- **Edges**: Spatial adjacency + feature similarity
- **Edge Weight**: $w_{ij} = \exp(-\frac{||f_i - f_j||^2}{2\sigma^2})$

In [ ]:
def find_adjacent_superpixels(segments):
    """
    Find spatially adjacent superpixels using 4-connectivity
    Returns: dict {seg_id: [adjacent_seg_ids]}
    """
    adjacency = {}
    h, w = segments.shape
    
    for i in range(h):
        for j in range(w):
            seg_id = segments[i, j]
            
            # Initialize if not seen
            if seg_id not in adjacency:
                adjacency[seg_id] = set()
            
            # Check 4 neighbors
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < h and 0 <= nj < w:
                    neighbor_id = segments[ni, nj]
                    if neighbor_id != seg_id:
                        adjacency[seg_id].add(neighbor_id)
    
    return adjacency


def build_adjacency_graph(segments, features, config):
    """
    Build graph with weighted edges based on:
    1. Spatial adjacency (4-connectivity)
    2. Feature similarity (Euclidean distance)
    
    Returns: NetworkX graph G with feature vectors and edge weights
    """
    # Find spatially adjacent superpixels
    adjacency = find_adjacent_superpixels(segments)
    
    # Initialize graph
    G = nx.Graph()
    
    # Add nodes with features
    n_segments = len(features)
    for seg_id in range(n_segments):
        G.add_node(seg_id, features=features[seg_id])
    
    # Compute pairwise distances for feature similarity
    dist_matrix = euclidean_distances(features)
    
    # Add edges based on spatial adjacency
    edges_added = 0
    for seg_id, adjacent_ids in adjacency.items():
        for adj_id in adjacent_ids:
            if seg_id < adj_id:  # Avoid duplicate edges
                # Compute edge weight: Gaussian kernel on feature distance
                feat_distance = dist_matrix[seg_id, adj_id]
                
                # Gaussian kernel: exp(-dist^2 / (2*sigma^2))
                weight = np.exp(-feat_distance**2 / (2 * config['sigma']**2))
                
                # Add edge only if weight is significant
                if weight > 0.01:  # Threshold to avoid very weak edges
                    G.add_edge(seg_id, adj_id, weight=weight, distance=feat_distance)
                    edges_added += 1
    
    return G, adjacency


print("✓ Graph construction functions defined")

## STAGE 4.4: Spectral Clustering Module

### Clustering Approach:
- Uses graph Laplacian: $L = D - A$ (degree matrix - adjacency matrix)
- Eigen decomposition separates disease from healthy regions
- **Output**: 3 clusters per image (diseased, healthy, uncertain)

In [ ]:
def perform_spectral_clustering(G, n_clusters=3):
    """
    Perform spectral clustering on the graph
    
    Args:
        G: NetworkX graph
        n_clusters: Number of clusters
    
    Returns:
        cluster_labels: (n_nodes,) array with cluster assignments
    """
    # Convert to adjacency matrix
    adj_matrix = nx.to_numpy_array(G)
    
    # Add small value to diagonal for numerical stability
    adj_matrix += np.eye(adj_matrix.shape[0]) * 0.01
    
    # Perform spectral clustering
    clustering = SpectralClustering(
        n_clusters=n_clusters,
        affinity='precomputed',
        assign_labels='kmeans',
        random_state=42,
        n_init=10
    )
    
    labels = clustering.fit_predict(adj_matrix)
    
    return labels


def cluster_graph_to_segmentation(segments, labels):
    """
    Convert cluster labels to segmentation mask
    
    Args:
        segments: (h, w) superpixel mask
        labels: (n_superpixels,) cluster assignments
    
    Returns:
        seg_mask: (h, w) cluster mask with values 0, 1, 2, ...
    """
    seg_mask = np.zeros_like(segments, dtype=np.uint8)
    
    for seg_id, cluster_id in enumerate(labels):
        seg_mask[segments == seg_id] = cluster_id
    
    return seg_mask


print("✓ Spectral clustering functions defined")

## STAGE 4.5: Superpixel Generation

### Generate superpixels using selected algorithm

In [ ]:
def generate_superpixels(img_rgb, config):
    """
    Generate superpixels using configured algorithm
    Returns: segments (h, w) with segment IDs
    """
    algorithm = config['superpixel_algorithm'].lower()
    
    if algorithm == 'slic':
        segments = slic(
            img_rgb,
            n_segments=config['num_segments'],
            compactness=config['compactness'],
            sigma=1,
            start_label=0
        )
    
    elif algorithm == 'felzenszwalb':
        segments = felzenszwalb(
            img_rgb,
            scale=config['felz_scale'],
            sigma=0.5,
            min_size=15
        )
    
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    return segments


print("✓ Superpixel generation function defined")